In [ ]:
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# =====================
# 1️⃣ Connect to Chroma
# =====================
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# =====================
# 2️⃣ Load collection
# =====================
collection_name = "member_resume_search_master_vector"
try:
    collection = client.get_collection(name=collection_name)
except Exception as e:
    print(f"❌ ไม่พบ collection ชื่อ '{collection_name}': {e}")
    raise

# =====================
# 3️⃣ Load embedding model
# =====================
embedder = SentenceTransformer("intfloat/multilingual-e5-large")

# =====================
# 4️⃣ Queries
# =====================
queries = [
    "microsoft office",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*80}")
    print(f"🔍 Query {i}: {query}")
    print(f"{'='*80}")

    # Encode query
    query_emb = embedder.encode([query]).tolist()

    # Query collection
    results = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        include=["documents", "metadatas", "distances"]
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    if docs:
        for j, (doc, meta, dist) in enumerate(zip(docs, metas, dists), 1):
            print(f"\n📋 อันดับ {j} (distance: {dist:.4f})")
            print(f"   🔹 id: {meta.get('id', 'N/A')}")
            print(f"   member_resume_id: {meta.get('member_resume_id', 'N/A')}")
            print(f"   member_user_id: {meta.get('member_user_id', 'N/A')}")
            print(f"   อาชีพหลัก: {meta.get('occupation_new_name', 'N/A')}")
            print(f"   อาชีพรอง: {meta.get('occupation_sub_name', 'N/A')}")
            print(f"   รายละเอียด: {doc[:150]}..." if len(doc) > 150 else f"   รายละเอียด: {doc}")
    else:
        print("   ❌ ไม่พบผลลัพธ์")
